#### Imports

In [95]:
import os
import time
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.language_models import LLM
from groq import Groq


#### Loading Env

In [82]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [83]:
from langchain_core.language_models import LLM
from groq import Groq

client = Groq(api_key=groq_api_key)
class GroqLLM(LLM):
    model: str = "llama-3.1-8b-instant"

    def _call(self, prompt, stop=None):

        if isinstance(prompt,list):
            messages= prompt
        else:
            messages=[
                {'role':"user", 'content': prompt}
            ]

        response = client.chat.completions.create(
            model = self.model,
            messages = messages
        )

        return response.choices[0].message.content
    
    @property
    def _llm_type(self):
        return"groq"

In [84]:
llm = GroqLLM()

#### Loading PDF

In [85]:
pdf_path = "data/book.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

documents = [doc for doc in documents if doc.metadata["page"] >= 17]

print(f"Total pages loaded:{len(documents)}")

Total pages loaded:834


In [86]:
print('Sample Content:\n')
print(documents[0].page_content[:500])

print('\nMetadata:\n')
print(documents[0].metadata)

Sample Content:

Perhaps you would like to give your homemade robot a brain of its own? Make it rec‐
ognize faces? Or learn to walk around?
Or maybe your company has tons of data (user logs, financial data, production data,
machine sensor data, hotline stats, HR reports, etc.), and more than likely you could
unearth some hidden gems if you just knew where to look. With Machine Learning,
you could accomplish the following and more:
• Segment customers and find the best marketing strategy for each group.
• Recomme

Metadata:

{'producer': 'calibre (4.8.0) [https://calibre-ebook.com]', 'creator': 'calibre (4.8.0) [https://calibre-ebook.com]', 'creationdate': '2019-10-10T14:06:12+00:00', 'author': 'Aurélien Géron', 'ebx_publisher': "O'Reilly Media, Incorporated", 'moddate': '2020-01-16T12:03:44+00:00', 'title': 'Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow', 'trapped': '/False', 'source': 'data/book.pdf', 'total_pages': 851, 'page': 17, 'page_label': 'xvi'}


#### Text Splitter

In [113]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 800,
    chunk_overlap = 200,
    separators=["\n\n", "\n", " ", ""] 
)

split_docs = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(split_docs)}")
for i in range(0,2):
    print("\n",split_docs[i].page_content[:800])


Total chunks created: 2898

 Perhaps you would like to give your homemade robot a brain of its own? Make it rec‐
ognize faces? Or learn to walk around?
Or maybe your company has tons of data (user logs, financial data, production data,
machine sensor data, hotline stats, HR reports, etc.), and more than likely you could
unearth some hidden gems if you just knew where to look. With Machine Learning,
you could accomplish the following and more:
• Segment customers and find the best marketing strategy for each group.
• Recommend products for each client based on what similar clients bought.
• Detect which transactions are likely to be fraudulent.
• Forecast next year’s revenue.
Whatever the reason, you have decided to learn Machine Learning and implement it
in your projects. Great idea!
Objective and Approach

 • Forecast next year’s revenue.
Whatever the reason, you have decided to learn Machine Learning and implement it
in your projects. Great idea!
Objective and Approach
This book assu

Creating Summary and Adding them

In [99]:

def generate_summary(text):
    prompt = f"""
Summarize the following text in ONE clear and specific sentence.
Include key examples if present. Do not be too generic.

TEXT:
{text}
"""
    return llm.invoke(prompt)

In [100]:
for i, doc in enumerate(split_docs):
    if i < 50:  
        doc.metadata["summary"] = generate_summary(doc.page_content)
        time.sleep(2)
    else:
        doc.metadata["summary"] = "Summary not generated"

In [103]:
for i in range(3):
    print("\nContent:\n", split_docs[i].page_content[:300])
    print("\nSummary:\n", split_docs[i].metadata["summary"])
    print('\n',split_docs[i].metadata)


Content:
 Perhaps you would like to give your homemade robot a brain of its own? Make it rec‐
ognize faces? Or learn to walk around?
Or maybe your company has tons of data (user logs, financial data, production data,
machine sensor data, hotline stats, HR reports, etc.), and more than likely you could
unearth

Summary:
 To implement Machine Learning, one can use it to accomplish a variety of tasks, such as segmenting customers and determining the best marketing strategy for each demographic group.

 {'producer': 'calibre (4.8.0) [https://calibre-ebook.com]', 'creator': 'calibre (4.8.0) [https://calibre-ebook.com]', 'creationdate': '2019-10-10T14:06:12+00:00', 'author': 'Aurélien Géron', 'ebx_publisher': "O'Reilly Media, Incorporated", 'moddate': '2020-01-16T12:03:44+00:00', 'title': 'Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow', 'trapped': '/False', 'source': 'data/book.pdf', 'total_pages': 851, 'page': 17, 'page_label': 'xvi', 'summary': 'To implement Machine L

#### Embeddings

In [104]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

sample_text = split_docs[0].page_content
embedding_vector = embedding_model.embed_query(sample_text)
print("Embedding Dimension:",len(embedding_vector))

C:\Users\vikas g\AppData\Local\Temp\ipykernel_20108\725833801.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
d:\RAG\venv\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Embedding Dimension: 384


#### Vector store

In [105]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding= embedding_model
)

print("Vector store created")

vectorstore.save_local("faiss_index_langchain")

Vector store created


In [106]:
vectorstore = FAISS.load_local(
    "faiss_index_langchain",
    embedding_model,
    allow_dangerous_deserialization=True
)

#### Retriever

In [107]:
retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs ={"k":5}
)

In [108]:
query = "What is overfitting?"

docs = retriever.invoke(query)
print(f"Retrieved {len(docs)} documents\n")

for i, doc in enumerate(docs):
    print(f"Document{i+1}")
    print(doc.page_content[:300])
    print("Page:", doc.metadata.get("page"))
    print()

Retrieved 5 documents

Document1
to training and remains constant during training. If you set the regularization hyper‐
parameter to a very large value, you will get an almost flat model (a slope close to
zero); the learning algorithm will almost certainly not overfit the training data, but it
will be less likely to find a good sol
Page: 58

Document2
performance. If a model performs well on the training data but generalizes poorly
according to the cross-validation metrics, then your model is overfitting. If it per‐
forms poorly on both, then it is underfitting. This is one way to tell when a model is
too simple or too complex.
Another way to tel
Page: 160

Document3
• Creating new features by gathering new data
Now that we have looked at many examples of bad data, let’s look at a couple of exam‐
ples of bad algorithms.
Overfitting  the Training Data
Say you are visiting a foreign country and the taxi driver rips you off. Y ou might be
tempted to say that all ta
Page: 56

Document4
par

#### Groq LLM Wrapper

In [109]:
from langchain_core.prompts import ChatPromptTemplate

prompt= ChatPromptTemplate.from_messages([
    ("system",
      "You are a strict RAG assistant. Answer ONLY from the provided context. "
     "If the answer is not found, say: 'I don't know based on the provided document.'" ),

    ("user",
      """Context:
        {context}

        Question:
        {question}

        Answer:
      """
    )
])

In [110]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = GroqLLM()

def format_docs(docs):
    return "\n\n ".join(
        f"""
        Source: Page {doc.metadata.get('page')}

        Content:
        {doc.page_content}
        """
    for doc in docs
    )

rag_chain =(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [112]:
query = "Explain relu in 5-6 points"

response = rag_chain.invoke(query)

print(response)

Based on the provided context from Page 321 and Page 364:

1. ReLU (Rectified Linear Unit) is an activation function that maps all negative values to 0 and all positive values to themselves. It is defined as ReLU(z) = max(0, z).

2. ReLU is continuous but not differentiable at z = 0, where the slope changes abruptly, which can make Gradient Descent bounce around.

3. Despite this limitation, ReLU functions very well in deep neural networks and has the advantage of being fast to compute.

4. ReLU does not have a maximum output value, which helps reduce the risk of vanishing gradients and makes it a good choice for deep networks.

5. The reason ReLU generally works better in ANNs is because it does not saturate for positive values, unlike sigmoid functions, and it's fast to compute, allowing it to be the default activation function.

I don't know based on the provided document whether ReLU's effectiveness can be attributed to other specific factors.
